# 🎯 Job / Internship Application Tracker Automation
## Ubuntu Automation for Data Science — Group Project (Scenario 5 of 5)

---

| Field | Detail |
|---|---|
| **Course** | Ubuntu Automation for Data Science |
| **Assignment** | Group Project — Job/Internship Application Tracker |
| **Scenario** | Scenario 5 — Automated Job Application Analytics |
| **Dataset** | `job_applications.csv` (400 rows, Jan–Apr 2024) |
| **Author** | [Shakhzodbek Jabbarov] |
| **Date** | May 2026 |

---

### 📋 Notebook Structure

| Part | Title | Description |
|---|---|---|
| Setup | Environment & Imports | Libraries, global config, constants |
| Part 1 | Data Loading & Cleaning | CSV ingestion, feature engineering, summaries |
| Part 2 | Status Dashboard | KPIs, donut chart, grouped bar by type |
| Part 3 | Follow-up Alert System | Overdue/Due Today/Upcoming digest + charts |
| Part 4 | Recruitment Funnel | Funnel analysis across multiple segments |
| Part 5 | Platform & Salary Intelligence | Platform stats, salary distributions |
| Part 6 | Timeline Analysis | Weekly trends, response-time histogram, heatmap |
| Part 7 | Automated Weekly Report | Full markdown report saved to file |
| Reflection | Personal Reflection | What I learned from this project |

---

> **How to run:** Upload `job_applications.csv` to the same directory, then run all cells top-to-bottom (`Runtime → Run all`).


## ⚙️ Setup — Imports, Styling, Constants

Before anything else we import every library we'll need, configure matplotlib's default style,
and define a global `TODAY` constant so every date-relative calculation stays reproducible.

**Why freeze TODAY?**  
Real automation scripts often run on a scheduler (cron).  
Freezing the reference date means the notebook produces identical output no matter when it is re-run,
which makes grading and peer-review fair.


In [ ]:
# ── Standard library imports ────────────────────────────────────────────────
import os                          # Operating-system utilities (path checks, etc.)
import re                          # Regular expressions (used in report generator)
import warnings                    # Suppress non-critical warnings

# ── Data manipulation ────────────────────────────────────────────────────────
import numpy as np                 # Numerical arrays and math helpers
import pandas as pd                # DataFrames — the backbone of every analysis

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib                  # Core matplotlib (needed for rcParams)
import matplotlib.pyplot as plt    # pyplot: the main plotting interface
import matplotlib.patches as mpatches  # For custom legend handles
import matplotlib.ticker as mticker    # Tick formatting helpers
import seaborn as sns              # Statistical plot shortcuts (heatmap, boxplot …)

# ── Suppress FutureWarning / UserWarning clutter ─────────────────────────────
warnings.filterwarnings("ignore")

print("=" * 60)
print("  All libraries imported successfully")
print("=" * 60)


### 🎨 Matplotlib Global Style

We set `rcParams` once here so every figure in the notebook inherits the same fonts,
DPI, and grid style — no need to repeat styling code in every plot cell.


In [ ]:
# ── Global matplotlib style ──────────────────────────────────────────────────
plt.rcParams["figure.dpi"]          = 130           # Crisp display in Colab
plt.rcParams["figure.facecolor"]    = "#FAFAFA"     # Soft off-white background
plt.rcParams["axes.facecolor"]      = "#FFFFFF"     # Pure white axes area
plt.rcParams["axes.spines.top"]     = False         # Remove top border
plt.rcParams["axes.spines.right"]   = False         # Remove right border
plt.rcParams["axes.grid"]           = True          # Show grid lines
plt.rcParams["grid.alpha"]          = 0.4           # Make grid subtle
plt.rcParams["grid.linestyle"]      = "--"          # Dashed grid
plt.rcParams["font.family"]         = "DejaVu Sans" # Readable sans-serif font
plt.rcParams["axes.titlesize"]      = 13            # Chart title font size
plt.rcParams["axes.labelsize"]      = 11            # Axis label font size
plt.rcParams["xtick.labelsize"]     = 9             # X-tick font size
plt.rcParams["ytick.labelsize"]     = 9             # Y-tick font size
plt.rcParams["legend.fontsize"]     = 9             # Legend text size
plt.rcParams["legend.framealpha"]   = 0.8           # Semi-transparent legend box

# ── Colour palette (used throughout) ─────────────────────────────────────────
PALETTE = sns.color_palette("Set2", 8)  # 8 distinct but harmonious colours

print("Matplotlib rcParams configured.")
print(f"Default DPI: {plt.rcParams['figure.dpi']}")


### 📅 Global Date Constant

`TODAY` acts as the notebook's "current date".  
We set it to **2024-04-10**, which is slightly after the last application in our dataset (April 9).


In [ ]:
# ── Freeze the reference date ─────────────────────────────────────────────────
TODAY = pd.Timestamp("2024-04-10")  # Simulate running the script on April 10 2024

print("=" * 60)
print(f"  Reference date (TODAY) = {TODAY.date()}")
print(f"  Day of week            = {TODAY.strftime('%A')}")
print(f"  ISO week number        = {TODAY.isocalendar().week}")
print("=" * 60)


---
## 📂 Part 1 — Data Loading & Cleaning

This section covers the full data ingestion pipeline:

1. **Load** the CSV into a pandas DataFrame
2. **Inspect** shape, dtypes, missing values
3. **Parse dates** from string columns
4. **Engineer new features** that power later analyses
5. **Print a summary block** confirming every category is present

### Why clean before analysing?
Raw CSVs from applicant-tracking systems (ATS) often contain:
- Date columns stored as plain strings
- Missing response dates for applications still pending
- Inconsistent category labels

Handling these issues upfront means every downstream cell can trust the data.


In [ ]:
# ── Read the CSV ──────────────────────────────────────────────────────────────
CSV_PATH = "job_applications.csv"          # File must be in the Colab working directory

df = pd.read_csv(CSV_PATH)                 # Load into a pandas DataFrame

# ── Quick shape report ────────────────────────────────────────────────────────
print("=" * 60)
print("  DATA LOAD REPORT")
print("=" * 60)
print(f"  Rows    : {df.shape[0]}")       # Number of application records
print(f"  Columns : {df.shape[1]}")       # Number of tracked fields
print("=" * 60)


### 👀 First Look at the Data

`df.head(8)` shows the first eight rows — enough to spot any obvious loading errors
(wrong delimiter, header repeated, extra blank columns, etc.).


In [ ]:
# ── Preview first 8 rows ──────────────────────────────────────────────────────
print("First 8 rows of the raw DataFrame:")
print("-" * 60)
df.head(8)                                 # Rendered as an HTML table in Colab


### 🔍 Column Data Types

Understanding dtypes matters because date arithmetic and groupby operations behave
differently on `object` vs `datetime64` columns.  
We'll convert the date columns next.


In [ ]:
# ── Inspect data types ────────────────────────────────────────────────────────
print("Column dtypes before date parsing:")
print("-" * 40)
print(df.dtypes)                           # Show each column's dtype
print()
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")


### 🗓️ Parse Date Columns

Three columns store dates as strings: `date_applied`, `follow_up_date`, `response_date`.
We convert them to `datetime64` so we can subtract them, extract week numbers, etc.

`errors='coerce'` turns unparseable values (e.g., empty strings) into `NaT`
(Not a Time) rather than raising an exception — safe for our partially-missing `response_date`.


In [ ]:
# ── Convert string columns → datetime ────────────────────────────────────────
df["date_applied"]   = pd.to_datetime(df["date_applied"],   errors="coerce")
df["follow_up_date"] = pd.to_datetime(df["follow_up_date"], errors="coerce")
df["response_date"]  = pd.to_datetime(df["response_date"],  errors="coerce")  # NaT where blank

print("Date columns converted to datetime64.")
print()

# ── Confirm new dtypes ────────────────────────────────────────────────────────
print("Updated dtypes for date columns:")
print(df[["date_applied", "follow_up_date", "response_date"]].dtypes)


### 📊 Missing Value Audit

Identifying nulls tells us:
- Which columns are complete (no action needed)
- Which need imputation or special handling in later cells

`response_date` should have the most nulls — applications that are still pending,
ghosted, or just applied won't have a response date yet.


In [ ]:
# ── Missing value counts ──────────────────────────────────────────────────────
missing = df.isnull().sum()                         # Count NaN/NaT per column
missing_pct = (missing / len(df) * 100).round(1)   # Convert to percentage

missing_report = pd.DataFrame({
    "Missing Count": missing,                        # Absolute count
    "Missing %": missing_pct                         # Percentage
}).query("`Missing Count` > 0")                     # Show only columns with gaps

print("=" * 50)
print("  MISSING VALUE REPORT")
print("=" * 50)
if len(missing_report) == 0:
    print("  No missing values found!")
else:
    print(missing_report.to_string())               # Print the full table
print("=" * 50)


### 🛠️ Feature Engineering

We derive seven new columns that power every later analysis.
Engineering features here — rather than inline — keeps later cells concise.

| New Column | Formula | Purpose |
|---|---|---|
| `days_to_response` | `response_date − date_applied` | How long each company took to reply |
| `salary_mid` | `(salary_min + salary_max) / 2` | Mid-point for fair salary comparisons |
| `month_applied` | `.dt.month_name()` | Grouping by calendar month |
| `week_applied` | `.dt.isocalendar().week` | Grouping by ISO week number |
| `days_since_applied` | `TODAY − date_applied` | Age of each application |
| `got_response` | `response_date.notna()` | Binary flag for response rate |
| `reached_interview` | `status ∈ {Interview, Offer}` | Binary flag for interview conversion |


In [ ]:
# ── Feature 1: days until company responded ────────────────────────────────────
df["days_to_response"] = (
    df["response_date"] - df["date_applied"]    # Timedelta: response minus applied
).dt.days                                       # Extract integer days

# ── Feature 2: salary midpoint ────────────────────────────────────────────────
df["salary_mid"] = (
    df["salary_min"] + df["salary_max"]         # Sum of band edges
) / 2                                           # Divide to get midpoint

# ── Feature 3: calendar month name ───────────────────────────────────────────
df["month_applied"] = df["date_applied"].dt.month_name()  # e.g. 'January'

# ── Feature 4: ISO week number ────────────────────────────────────────────────
df["week_applied"] = df["date_applied"].dt.isocalendar().week.astype(int)

# ── Feature 5: how many days ago was this application submitted ───────────────
df["days_since_applied"] = (
    TODAY - df["date_applied"]                  # Timedelta from apply date to today
).dt.days                                       # Extract integer days

# ── Feature 6: did the company ever send a response? (boolean) ────────────────
df["got_response"] = df["response_date"].notna()  # True if response_date is not NaT

# ── Feature 7: did the application reach interview stage or higher? ───────────
df["reached_interview"] = df["status"].isin(["Interview", "Offer"])

print("Feature engineering complete.")
print(f"DataFrame now has {df.shape[1]} columns (was 19).")


### 📈 Full Dataset Summary

The summary block below prints counts for every categorical variable so we can
verify the simulation produced a realistic distribution.

> **Healthy signs to look for:**
> - Offer rate ≈ 10–15%
> - Ghosted rate ≈ 15–20%
> - Applied (still pending) ≈ 10–20%


In [ ]:
# ── Helper: count + percentage table ─────────────────────────────────────────
def count_pct(series, title):
    """Print value counts with percentages for a categorical Series."""
    vc = series.value_counts()                       # Raw counts
    pct = (vc / len(df) * 100).round(1)              # Percentage of total rows
    print(f"\n{'─'*40}")
    print(f"  {title}")
    print(f"{'─'*40}")
    for val, cnt in vc.items():
        print(f"  {str(val):<25} {cnt:>4}  ({pct[val]:>5.1f}%)")

print("=" * 60)
print("  FULL DATASET SUMMARY")
print("=" * 60)
print(f"  Total applications  : {len(df)}")
print(f"  Date range          : {df['date_applied'].min().date()} → {df['date_applied'].max().date()}")
print(f"  Unique companies    : {df['company'].nunique()}")
print(f"  Unique roles        : {df['role'].nunique()}")

count_pct(df["status"],          "Status Distribution")
count_pct(df["type"],            "Job Type (Internship vs Full-time)")
count_pct(df["platform"],        "Application Platform")
count_pct(df["resume_version"],  "Resume Version Used")
count_pct(df["cover_letter"],    "Cover Letter Submitted")
count_pct(df["referral"],        "Used Referral")
count_pct(df["remote"],          "Remote Position")

print("\n" + "=" * 60)


### 🔢 Descriptive Statistics (`df.describe()`)

`describe()` shows min/max/mean/std for every numeric column — useful for
spotting outliers (e.g., a salary_max of 0) or impossible values (negative days).


In [ ]:
# ── Numerical summary ─────────────────────────────────────────────────────────
print("Descriptive statistics for numeric columns:")
print("-" * 60)
df.describe().round(1)                             # Round to 1 decimal for readability


In [ ]:
# ── Top 10 most-applied companies ─────────────────────────────────────────────
print("Top 10 companies by number of applications:")
print("-" * 45)
top10 = df["company"].value_counts().head(10)       # Count applications per company
for company, count in top10.items():
    bar = "█" * count                               # ASCII bar proportional to count
    print(f"  {company:<25} {count:>3}  {bar}")


---
## 📊 Part 2 — Status Dashboard

The Status Dashboard is the first thing a job-seeker would check each morning.
It answers three questions at a glance:

1. **What is my overall offer rate?**
2. **How often am I reaching interview stage?**
3. **What fraction of companies are ghosting me?**

We visualise the status breakdown two ways:
- A **donut chart** — great for part-to-whole relationships
- A **grouped bar chart** — shows how status splits across Internship vs Full-time

### Colour coding convention
Throughout the notebook, status colours stay consistent:

| Status | Colour |
|---|---|
| Offer | 🟢 `#2ecc71` |
| Interview | 🔵 `#3498db` |
| Applied | 🟡 `#f1c40f` |
| Rejected | 🔴 `#e74c3c` |
| Ghosted | ⚫ `#95a5a6` |


In [ ]:
# ── Define consistent colours for each status ─────────────────────────────────
STATUS_COLORS = {
    "Offer":     "#2ecc71",   # Green  — positive outcome
    "Interview": "#3498db",   # Blue   — in progress
    "Applied":   "#f1c40f",   # Yellow — waiting
    "Rejected":  "#e74c3c",   # Red    — negative outcome
    "Ghosted":   "#95a5a6",   # Grey   — no response
}

print("STATUS_COLORS dictionary defined:")
for status, color in STATUS_COLORS.items():
    print(f"  {status:<12} → {color}")


### 📦 KPI Box — Key Performance Indicators

Before drawing charts, we print the three headline metrics.
This is useful for automated reports (Part 7) or pasting into a daily standup update.


In [ ]:
# ── Compute KPIs ──────────────────────────────────────────────────────────────
total_apps     = len(df)                              # Total applications submitted

# Count each status group
offer_count      = (df["status"] == "Offer").sum()         # Number of offers
interview_count  = (df["status"] == "Interview").sum()     # Active interviews
rejected_count   = (df["status"] == "Rejected").sum()      # Rejections
ghosted_count    = (df["status"] == "Ghosted").sum()       # No response received
applied_count    = (df["status"] == "Applied").sum()       # Still pending

# Compute rates as percentages
offer_rate      = offer_count     / total_apps * 100   # % of apps that led to offer
interview_rate  = interview_count / total_apps * 100   # % of apps that reached interview
ghosted_rate    = ghosted_count   / total_apps * 100   # % of apps that got no reply
response_rate   = df["got_response"].mean()   * 100    # % of apps with any reply

print("=" * 55)
print("  📊  JOB APPLICATION KPI DASHBOARD")
print("=" * 55)
print(f"  Total Applications  :  {total_apps:>4}")
print(f"  ─────────────────────────────────────")
print(f"  ✅  Offers           :  {offer_count:>4}  ({offer_rate:.1f}%)")
print(f"  💬  Interviews       :  {interview_count:>4}  ({interview_rate:.1f}%)")
print(f"  ❌  Rejected         :  {rejected_count:>4}  ({rejected_count/total_apps*100:.1f}%)")
print(f"  👻  Ghosted          :  {ghosted_count:>4}  ({ghosted_rate:.1f}%)")
print(f"  ⏳  Still Pending    :  {applied_count:>4}  ({applied_count/total_apps*100:.1f}%)")
print(f"  ─────────────────────────────────────")
print(f"  📬  Response Rate    :  {response_rate:.1f}%")
print("=" * 55)


### 🍩 Donut Chart + Grouped Bar Chart

Two complementary views of the status data:

- **Left (donut):** Overall proportion of each status across all 400 applications
- **Right (grouped bar):** Same statuses broken out by Internship vs Full-time

**Reading the grouped bar:** if Full-time offers are a taller bar than Internship offers,
it tells you that full-time applications have a higher offer conversion rate.


In [ ]:
# ── Figure setup: 1 row, 2 columns ───────────────────────────────────────────
fig, (ax_donut, ax_bar) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Application Status Overview", fontsize=15, fontweight="bold", y=1.01)

# ── LEFT: Donut chart ─────────────────────────────────────────────────────────
status_counts = df["status"].value_counts()              # Count each status
donut_colors  = [STATUS_COLORS[s] for s in status_counts.index]  # Match colours

wedges, texts, autotexts = ax_donut.pie(
    status_counts.values,                   # Slice sizes
    labels=status_counts.index,             # Category labels
    colors=donut_colors,                    # Per-status colour
    autopct="%1.1f%%",                      # Show percentage inside each slice
    startangle=140,                         # Rotate so first slice is at top-left
    wedgeprops=dict(width=0.55,             # Donut hole width (0 = pie, 1 = ring)
                    edgecolor="white",      # White border between slices
                    linewidth=2),
    pctdistance=0.75,                       # Push % labels toward rim
)
for at in autotexts:
    at.set_fontsize(8)                      # Smaller font so labels don't overlap
ax_donut.set_title("Overall Status Breakdown", fontsize=12, pad=12)

# Centre annotation showing total count
ax_donut.text(0, 0, f"{total_apps}\napps",
              ha="center", va="center", fontsize=11, fontweight="bold")

# ── RIGHT: Grouped bar chart (status × job type) ──────────────────────────────
cross_tab = pd.crosstab(df["status"], df["type"])     # Rows=status, cols=job type

x_pos   = np.arange(len(cross_tab))                  # One cluster per status
bar_w   = 0.35                                        # Width of each bar
colors2 = ["#5b9bd5", "#ed7d31"]                      # Blue=Internship, Orange=Full-time

for i, col in enumerate(cross_tab.columns):           # Iterate over job types
    ax_bar.bar(
        x_pos + i * bar_w,                            # Offset so bars don't overlap
        cross_tab[col].values,                        # Bar heights
        width=bar_w,                                  # Bar width
        label=col,                                    # Legend label
        color=colors2[i],                             # Bar colour
        edgecolor="white",                            # White border
        linewidth=0.8,
    )

ax_bar.set_xticks(x_pos + bar_w / 2)                 # Centre tick under each group
ax_bar.set_xticklabels(cross_tab.index, rotation=15) # Status labels at slight angle
ax_bar.set_title("Status by Job Type", fontsize=12)
ax_bar.set_ylabel("Number of Applications")
ax_bar.legend(title="Job Type")

plt.tight_layout()
plt.savefig("status_dashboard.png", dpi=130, bbox_inches="tight")
plt.show()
print("Figure saved → status_dashboard.png")


---
## 🔔 Part 3 — Follow-up Alert System

Many job seekers lose offers simply by forgetting to follow up.
This part automates that reminder process by:

1. **Filtering** to active (non-terminal) applications
2. **Computing** how many days remain until the suggested follow-up date
3. **Categorising** each application into an alert level
4. **Printing** a formatted digest grouped by urgency
5. **Visualising** the alert distribution and application ages

### Alert Level Logic

| Level | Condition | Meaning |
|---|---|---|
| 🔴 OVERDUE | `days_to_followup < 0` | Follow-up date has already passed |
| 🟡 DUE TODAY | `days_to_followup == 0` | Follow up today |
| 🟢 UPCOMING | `days_to_followup ∈ [1, 7]` | Follow up within the next week |
| ⚪ LATER | `days_to_followup > 7` | No immediate action needed |


In [ ]:
# ── Filter to active applications only ────────────────────────────────────────
# "Active" means the application hasn't reached a terminal state (Offer/Rejected/Ghosted)
active_statuses = ["Applied", "Interview"]            # Statuses still in play
df_active = df[df["status"].isin(active_statuses)].copy()  # Subset of DataFrame

print(f"Active applications (Applied + Interview): {len(df_active)}")
print(f"  → Still in 'Applied' stage : {(df_active['status']=='Applied').sum()}")
print(f"  → In 'Interview' stage     : {(df_active['status']=='Interview').sum()}")


In [ ]:
# ── Compute days remaining until follow-up date ───────────────────────────────
df_active["days_to_followup"] = (
    df_active["follow_up_date"] - TODAY            # Timedelta: follow-up minus today
).dt.days                                          # Convert to integer days

print("days_to_followup sample (first 10 rows):")
print(df_active[["application_id", "company", "status",
                  "follow_up_date", "days_to_followup"]].head(10).to_string(index=False))


### 🚦 Alert Level Function

We define `alert_level()` as a pure function so it's easy to unit-test and reuse
in the report generator (Part 7).


In [ ]:
# ── Define alert level categorisation ────────────────────────────────────────
def alert_level(days):
    """
    Classify days-until-follow-up into a priority level.

    Parameters
    ----------
    days : int  — days until follow-up date (negative = overdue)

    Returns
    -------
    str — one of: 'OVERDUE', 'DUE TODAY', 'UPCOMING', 'LATER'
    """
    if days < 0:                  # Follow-up date has already passed
        return "OVERDUE"
    elif days == 0:               # Follow-up is due today
        return "DUE TODAY"
    elif days <= 7:               # Follow-up due within the next 7 days
        return "UPCOMING"
    else:                         # More than a week away — no rush
        return "LATER"

# ── Apply function to every active application ────────────────────────────────
df_active["alert"] = df_active["days_to_followup"].apply(alert_level)

# ── Quick distribution check ──────────────────────────────────────────────────
print("Alert level distribution:")
print(df_active["alert"].value_counts().to_string())


### 📋 Formatted Alert Digest

The digest below mimics what you might receive as a Slack notification or email
from an automated daily cron job.  
Applications are sorted within each level by `days_to_followup` (most urgent first).


In [ ]:
# ── Alert level display order and emoji mapping ───────────────────────────────
ALERT_ORDER  = ["OVERDUE", "DUE TODAY", "UPCOMING", "LATER"]
ALERT_EMOJI  = {"OVERDUE": "🔴", "DUE TODAY": "🟡", "UPCOMING": "🟢", "LATER": "⚪"}

print("=" * 65)
print("  📬  DAILY FOLLOW-UP ALERT DIGEST")
print(f"  Generated: {TODAY.strftime('%A, %B %d %Y')}")
print("=" * 65)

for level in ALERT_ORDER:                                # Process in priority order
    subset = df_active[df_active["alert"] == level]      # Apps in this alert level
    if len(subset) == 0:                                 # Skip empty levels
        continue

    # Sort by urgency (most overdue first, earliest upcoming first)
    subset = subset.sort_values("days_to_followup")

    emoji = ALERT_EMOJI[level]                           # Get level emoji
    print(f"\n{emoji}  {level}  ({len(subset)} applications)")
    print("  " + "─" * 60)

    # Print each application's details
    for _, row in subset.iterrows():
        days_str = (
            f"{abs(int(row['days_to_followup']))} days ago"
            if row["days_to_followup"] < 0
            else f"in {int(row['days_to_followup'])} days"
            if row["days_to_followup"] > 0
            else "TODAY"
        )
        print(f"  • {row['company']:<22} │ {row['role'][:30]:<30} │ Follow up {days_str}")

print("\n" + "=" * 65)


### 📊 Alert Level Charts

Two complementary visualisations:

- **Left bar chart:** Number of applications per alert level
- **Right horizontal bar:** Days-since-applied per company (for active apps), sorted ascending
  — longest-waiting applications appear at the top, which helps prioritise outreach


In [ ]:
# ── Figure: 2 side-by-side charts ─────────────────────────────────────────────
fig, (ax_alert, ax_age) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Follow-up Alert System", fontsize=14, fontweight="bold")

# ── LEFT: Alert level bar chart ───────────────────────────────────────────────
alert_counts = df_active["alert"].value_counts().reindex(ALERT_ORDER, fill_value=0)
alert_colors = {"OVERDUE": "#e74c3c", "DUE TODAY": "#f39c12",
                "UPCOMING": "#2ecc71", "LATER": "#bdc3c7"}
bar_colors_list = [alert_colors[a] for a in alert_counts.index]

ax_alert.bar(
    alert_counts.index,          # X-axis: alert level names
    alert_counts.values,         # Bar heights
    color=bar_colors_list,       # Per-level colour
    edgecolor="white",           # White border
    linewidth=1,
)
ax_alert.set_title("Applications by Alert Level")
ax_alert.set_xlabel("Alert Level")
ax_alert.set_ylabel("Number of Applications")

# Add count labels on top of each bar
for i, (level, val) in enumerate(alert_counts.items()):
    ax_alert.text(i, val + 0.3, str(val),      # Position just above bar
                  ha="center", fontsize=9, fontweight="bold")

# ── RIGHT: Days since applied per company (active apps only) ──────────────────
company_age = (
    df_active.groupby("company")["days_since_applied"]
    .mean()                              # Average age per company
    .sort_values(ascending=True)         # Shortest waits at bottom
    .head(20)                            # Limit to top 20 for readability
)

ax_age.barh(
    company_age.index,           # Y-axis: company names
    company_age.values,          # Bar lengths
    color="#3498db",             # Consistent blue
    edgecolor="white",
)
ax_age.set_title("Avg Days Since Applied\n(Active Applications, Top 20)")
ax_age.set_xlabel("Days Since Applied")
ax_age.axvline(company_age.mean(),           # Vertical mean line
               color="red", linestyle="--",
               linewidth=1.2, label=f"Mean: {company_age.mean():.0f}d")
ax_age.legend(fontsize=8)

plt.tight_layout()
plt.savefig("followup_alerts.png", dpi=130, bbox_inches="tight")
plt.show()
print("Figure saved → followup_alerts.png")


---
## 🏔️ Part 4 — Recruitment Funnel Analysis

A recruitment funnel tracks how applications progress through stages.
We analyse **seven segments** to discover where attrition is highest
and which strategies improve conversion:

| Segment | Question |
|---|---|
| Overall | What is my baseline conversion? |
| Internship | Are intern apps better or worse than full-time? |
| Full-time | — (compare with above) |
| With Referral | How much does a referral help? |
| No Referral | — (compare with above) |
| Cover Letter | Does submitting a cover letter improve outcomes? |
| No Cover Letter | — (compare with above) |

### Funnel Stages (in order)
`Applied → Got Response → Reached Interview → Received Offer`


In [ ]:
# ── Funnel stage flags ────────────────────────────────────────────────────────
# Stage 1: application submitted (every row counts)
# Stage 2: company responded at all
# Stage 3: reached interview or received offer
# Stage 4: received offer

def funnel(subset):
    """
    Compute recruitment funnel metrics for a given DataFrame subset.

    Parameters
    ----------
    subset : pd.DataFrame  — slice of the main df

    Returns
    -------
    dict with keys: n, responded, interviewed, offered,
                    respond_rate, interview_rate, offer_rate
    """
    n           = len(subset)                                   # Total in segment
    responded   = subset["got_response"].sum()                  # Stage 2 count
    interviewed = subset["reached_interview"].sum()             # Stage 3 count
    offered     = (subset["status"] == "Offer").sum()           # Stage 4 count

    return {
        "n"              : n,
        "responded"      : responded,
        "interviewed"    : interviewed,
        "offered"        : offered,
        "respond_rate"   : responded   / n   * 100 if n   > 0 else 0,
        "interview_rate" : interviewed / n   * 100 if n   > 0 else 0,
        "offer_rate"     : offered     / n   * 100 if n   > 0 else 0,
    }

print("funnel() function defined and ready.")


In [ ]:
# ── Compute funnel for all seven segments ─────────────────────────────────────
segments = {
    "Overall"        : df,
    "Internship"     : df[df["type"] == "Internship"],
    "Full-time"      : df[df["type"] == "Full-time"],
    "With Referral"  : df[df["referral"] == "Yes"],
    "No Referral"    : df[df["referral"] == "No"],
    "Cover Letter"   : df[df["cover_letter"] == "Yes"],
    "No Cover Letter": df[df["cover_letter"] == "No"],
}

funnel_results = {name: funnel(subset) for name, subset in segments.items()}

# ── Print conversion rate table ───────────────────────────────────────────────
print("=" * 75)
print(f"  {'Segment':<20} {'N':>5}  {'Response%':>10}  {'Interview%':>11}  {'Offer%':>8}")
print("─" * 75)
for name, f in funnel_results.items():
    print(f"  {name:<20} {f['n']:>5}  {f['respond_rate']:>9.1f}%  "
          f"{f['interview_rate']:>10.1f}%  {f['offer_rate']:>7.1f}%")
print("=" * 75)


### 📊 Funnel Visualisation — 3 Side-by-Side Charts

Each chart displays the three conversion rates (Response %, Interview %, Offer %)
for a different pair of segments:

1. **Overall vs Internship vs Full-time**
2. **Referral: With vs Without**
3. **Cover Letter: With vs Without**


In [ ]:
# ── Build funnel chart data ───────────────────────────────────────────────────
metrics = ["respond_rate", "interview_rate", "offer_rate"]   # Three funnel stages
metric_labels = ["Response %", "Interview %", "Offer %"]     # Human-readable labels
bar_positions = np.arange(len(metrics))                      # x-coordinates [0, 1, 2]

# ── Create 1×3 figure ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.suptitle("Recruitment Funnel — Segment Comparison", fontsize=14, fontweight="bold")

# ── Funnel chart helper ────────────────────────────────────────────────────────
def plot_funnel_group(ax, seg_names, title, palette):
    """Draw a grouped bar funnel chart on the given axis."""
    n_segs = len(seg_names)                           # Number of segments to compare
    w = 0.8 / n_segs                                  # Bar width per segment

    for i, seg in enumerate(seg_names):
        vals = [funnel_results[seg][m] for m in metrics]  # Rates for this segment
        offset = (i - n_segs / 2 + 0.5) * w              # Centre the cluster
        bars = ax.bar(bar_positions + offset, vals,
                      width=w * 0.9,                      # Slight gap between bars
                      label=seg,
                      color=palette[i],
                      edgecolor="white")
        # Add value labels on top of each bar
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.4,
                    f"{val:.1f}%",
                    ha="center", va="bottom", fontsize=7)

    ax.set_xticks(bar_positions)
    ax.set_xticklabels(metric_labels)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Conversion Rate (%)")
    ax.legend(fontsize=8)
    ax.set_ylim(0, max([funnel_results[s][m] for s in seg_names for m in metrics]) * 1.25 + 5)

# ── Chart 1: Overall vs Internship vs Full-time ───────────────────────────────
plot_funnel_group(axes[0], ["Overall", "Internship", "Full-time"],
                 "By Job Type", ["#5b9bd5", "#ed7d31", "#a9d18e"])

# ── Chart 2: Referral comparison ──────────────────────────────────────────────
plot_funnel_group(axes[1], ["With Referral", "No Referral"],
                 "Referral Impact", ["#2ecc71", "#e74c3c"])

# ── Chart 3: Cover letter comparison ─────────────────────────────────────────
plot_funnel_group(axes[2], ["Cover Letter", "No Cover Letter"],
                 "Cover Letter Impact", ["#3498db", "#95a5a6"])

plt.tight_layout()
plt.savefig("funnel_comparison.png", dpi=130, bbox_inches="tight")
plt.show()
print("Figure saved → funnel_comparison.png")


### 📊 Rejection Stage & Referral Outcome Charts

Understanding *where* in the process applications get rejected helps focus improvement efforts:

- **Rejected at Resume Screening** → improve resume keywords / ATS optimisation
- **Rejected at Phone Screen** → practise elevator pitch and screening questions
- **Rejected at Final Round** → work on offer negotiation and final-round prep


In [ ]:
# ── Figure 2: Rejection stage + referral outcome ──────────────────────────────
fig, (ax_rej, ax_ref) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Rejection Analysis & Referral Outcomes", fontsize=14, fontweight="bold")

# ── LEFT: Rejection stage distribution ───────────────────────────────────────
rej_stage = (
    df[df["status"] == "Rejected"]          # Only rejected applications
    ["rejection_stage"]
    .value_counts()                          # Count by stage
    .sort_values(ascending=True)             # Shortest bar at bottom
)

ax_rej.barh(rej_stage.index, rej_stage.values,
            color="#e74c3c", edgecolor="white", alpha=0.85)
ax_rej.set_title("Where Applications Get Rejected")
ax_rej.set_xlabel("Number of Rejections")
for i, (val, label) in enumerate(zip(rej_stage.values, rej_stage.index)):
    ax_rej.text(val + 0.3, i, str(val), va="center", fontsize=9)

# ── RIGHT: Referral × status grouped bar ─────────────────────────────────────
ref_cross = pd.crosstab(df["status"], df["referral"])    # Status rows, referral cols

x2      = np.arange(len(ref_cross))                      # One cluster per status
w2      = 0.35                                            # Bar width
colors3 = ["#2ecc71", "#e74c3c"]                          # Green=Yes, Red=No

for i, col in enumerate(ref_cross.columns):               # 'No' and 'Yes'
    ax_ref.bar(
        x2 + i * w2,
        ref_cross[col].values,
        width=w2,
        label=f"Referral: {col}",
        color=colors3[i],
        edgecolor="white",
    )

ax_ref.set_xticks(x2 + w2 / 2)
ax_ref.set_xticklabels(ref_cross.index, rotation=15)
ax_ref.set_title("Status by Referral")
ax_ref.set_ylabel("Number of Applications")
ax_ref.legend()

plt.tight_layout()
plt.savefig("rejection_referral.png", dpi=130, bbox_inches="tight")
plt.show()
print("Figure saved → rejection_referral.png")


---
## 💰 Part 5 — Platform & Salary Intelligence

Job boards are not created equal — some platforms yield higher response rates,
better salaries, or more offers.  
This section answers:

- **Which platform has the highest response rate?**
- **Which platform lists the highest-paying roles?**
- **Which resume version performs best?**
- **How do salaries compare between Internships and Full-time roles?**

### Platforms in our dataset
| Platform | Notes |
|---|---|
| LinkedIn | Global professional network; common for MNCs |
| Wanted | Korean IT/startup-focused job board |
| Jumpit | Korean tech-focused, popular for developers |
| Company Website | Direct application — no recruiter middleman |
| Saramin | Large Korean general job portal |
| Jobkorea | Another large Korean general portal |


In [ ]:
# ── Compute per-platform aggregates ──────────────────────────────────────────
platform_stats = df.groupby("platform").agg(
    total_apps      = ("application_id", "count"),       # Applications per platform
    response_count  = ("got_response",   "sum"),         # Responses per platform
    offer_count     = ("status",         lambda x: (x=="Offer").sum()),  # Offers
    avg_salary_mid  = ("salary_mid",     "mean"),        # Average mid salary
    avg_response_d  = ("days_to_response","mean"),       # Mean days to respond
).reset_index()

# ── Derive rate columns ────────────────────────────────────────────────────────
platform_stats["response_rate"] = (
    platform_stats["response_count"] / platform_stats["total_apps"] * 100
).round(1)

platform_stats["offer_rate"] = (
    platform_stats["offer_count"] / platform_stats["total_apps"] * 100
).round(1)

platform_stats["avg_salary_mid"] = (
    platform_stats["avg_salary_mid"] / 10000                # Convert KRW to 만원
).round(0).astype(int)

platform_stats["avg_response_d"] = platform_stats["avg_response_d"].round(1)

# ── Print the full platform intelligence table ─────────────────────────────────
print("=" * 80)
print("  PLATFORM INTELLIGENCE TABLE (salary in 만원 / month)")
print("=" * 80)
print(platform_stats.to_string(index=False))
print("=" * 80)


### 📊 Platform & Salary Visualisation — 2×2 Figure

Four panels packed into one figure:

1. **Response Rate by Platform** — which platform actually gets replies?
2. **Average Salary by Platform** — which platform posts highest-paying roles?
3. **Salary Box Plot by Job Type** — distribution of salary_mid for Intern vs Full-time
4. **Resume Version Performance** — offer rate and interview rate per version


In [ ]:
# ── Build 2×2 figure ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Platform & Salary Intelligence", fontsize=15, fontweight="bold")

# ── Top-Left: Response rate by platform ───────────────────────────────────────
sorted_rr = platform_stats.sort_values("response_rate", ascending=False)
axes[0, 0].bar(sorted_rr["platform"], sorted_rr["response_rate"],
               color="#3498db", edgecolor="white")
axes[0, 0].set_title("Response Rate by Platform (%)")
axes[0, 0].set_xlabel("Platform")
axes[0, 0].set_ylabel("Response Rate (%)")
axes[0, 0].tick_params(axis="x", rotation=20)
for i, val in enumerate(sorted_rr["response_rate"]):
    axes[0, 0].text(i, val + 0.3, f"{val:.1f}%",
                    ha="center", fontsize=8, fontweight="bold")

# ── Top-Right: Average salary by platform ─────────────────────────────────────
sorted_sal = platform_stats.sort_values("avg_salary_mid", ascending=False)
axes[0, 1].bar(sorted_sal["platform"], sorted_sal["avg_salary_mid"],
               color="#2ecc71", edgecolor="white")
axes[0, 1].set_title("Avg Mid Salary by Platform (만원/month)")
axes[0, 1].set_xlabel("Platform")
axes[0, 1].set_ylabel("Avg Salary Mid (만원)")
axes[0, 1].tick_params(axis="x", rotation=20)
for i, val in enumerate(sorted_sal["avg_salary_mid"]):
    axes[0, 1].text(i, val + 0.5, f"{val}만",
                    ha="center", fontsize=8, fontweight="bold")

# ── Bottom-Left: Salary boxplot by job type ────────────────────────────────────
# Group salary_mid values into a list per type for boxplot
intern_sal = df.loc[df["type"]=="Internship",  "salary_mid"] / 10000  # 만원
ft_sal     = df.loc[df["type"]=="Full-time",   "salary_mid"] / 10000

bp = axes[1, 0].boxplot(
    [intern_sal.dropna(), ft_sal.dropna()],   # Two groups
    labels=["Internship", "Full-time"],        # X-tick labels
    patch_artist=True,                         # Fill boxes with colour
    medianprops=dict(color="black", linewidth=2),
)
bp["boxes"][0].set_facecolor("#aed6f1")        # Light blue for internship
bp["boxes"][1].set_facecolor("#a9dfbf")        # Light green for full-time
axes[1, 0].set_title("Salary Distribution by Job Type (만원/month)")
axes[1, 0].set_ylabel("Salary Mid (만원)")

# ── Bottom-Right: Resume version performance ──────────────────────────────────
resume_perf = df.groupby("resume_version").agg(
    offer_rate     = ("status", lambda x: (x=="Offer").sum() / len(x) * 100),
    interview_rate = ("reached_interview", "mean")
).reset_index()
resume_perf["interview_rate"] *= 100       # Convert fraction to percentage

x3 = np.arange(len(resume_perf))           # x positions
w3 = 0.35                                  # Bar width

axes[1, 1].bar(x3 - w3/2, resume_perf["interview_rate"],
               width=w3, label="Interview %", color="#5b9bd5", edgecolor="white")
axes[1, 1].bar(x3 + w3/2, resume_perf["offer_rate"],
               width=w3, label="Offer %",     color="#ed7d31", edgecolor="white")
axes[1, 1].set_xticks(x3)
axes[1, 1].set_xticklabels(resume_perf["resume_version"])
axes[1, 1].set_title("Performance by Resume Version")
axes[1, 1].set_ylabel("Rate (%)")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("platform_salary.png", dpi=130, bbox_inches="tight")
plt.show()
print("Figure saved → platform_salary.png")


---
## 📅 Part 6 — Timeline Analysis

Timing matters in job searching.  
This section reveals:

1. **Weekly application volume** — are you applying consistently or in bursts?
2. **Interviews and offers over time** — is effort translating to results on a lag?
3. **Response time distribution** — what is the typical wait for a reply?
4. **Month × Status heatmap** — which months were most productive?

### Why timeline analysis matters
- **Bursts** of applications (30 in one week, none the next) are less effective than steady outreach
- **Lag between apply date and response** helps set realistic expectations
- **Seasonal patterns** (e.g., Q1 hiring freeze at some companies) are visible in the heatmap


In [ ]:
# ── Weekly aggregation ────────────────────────────────────────────────────────
weekly = df.groupby("week_applied").agg(
    applied     = ("application_id",   "count"),           # Total applied that week
    interviews  = ("reached_interview","sum"),              # Interviews that week
    offers      = ("status",           lambda x: (x=="Offer").sum()),     # Offers
    rejections  = ("status",           lambda x: (x=="Rejected").sum()),  # Rejections
).reset_index()

print("Weekly aggregation preview:")
print(weekly.to_string(index=False))


### 📈 Weekly Trend Line Chart

The shaded area under the 'Applied' line helps visualise application volume.
Vertical reference lines would mark any career fairs or campus recruitment events
(you'd add those manually for a real tracker).


In [ ]:
# ── Line chart: weekly application activity ───────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

# Plot each metric as a line
ax.plot(weekly["week_applied"], weekly["applied"],
        marker="o", linewidth=2.2, color="#3498db", label="Applied")
ax.plot(weekly["week_applied"], weekly["interviews"],
        marker="s", linewidth=2.0, color="#f39c12", label="Interviews")
ax.plot(weekly["week_applied"], weekly["offers"],
        marker="^", linewidth=2.0, color="#2ecc71", label="Offers", markersize=8)
ax.plot(weekly["week_applied"], weekly["rejections"],
        marker="x", linewidth=1.5, color="#e74c3c", label="Rejections", linestyle="--")

# Shade the area under the 'applied' line
ax.fill_between(weekly["week_applied"], weekly["applied"],
                alpha=0.12, color="#3498db")

ax.set_title("Weekly Job Application Activity (Jan–Apr 2024)", fontsize=13)
ax.set_xlabel("ISO Week Number")
ax.set_ylabel("Number of Applications")
ax.legend()
ax.set_xticks(weekly["week_applied"])               # Show all week numbers

plt.tight_layout()
plt.savefig("weekly_timeline.png", dpi=130, bbox_inches="tight")
plt.show()
print("Figure saved → weekly_timeline.png")


### 📊 Response Time Histogram

The histogram shows how many days it typically takes companies to respond.
A long right tail indicates some companies take much longer than average —
those might benefit from an earlier follow-up.


In [ ]:
# ── Histogram of days_to_response ────────────────────────────────────────────
resp_data = df["days_to_response"].dropna()          # Drop NaT rows (ghosted / pending)
mean_resp = resp_data.mean()                          # Calculate mean for reference line

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    resp_data,
    bins=25,                                          # 25 equally-spaced bins
    color="#9b59b6",                                  # Purple bars
    edgecolor="white",                                # White border between bars
    alpha=0.85,
)
ax.axvline(mean_resp,                                 # Vertical mean reference line
           color="#e74c3c",
           linewidth=2,
           linestyle="--",
           label=f"Mean = {mean_resp:.1f} days")

ax.set_title("Distribution of Days to Response", fontsize=13)
ax.set_xlabel("Days from Application → First Response")
ax.set_ylabel("Number of Applications")
ax.legend()

# Annotate the mean line
ax.text(mean_resp + 0.5, ax.get_ylim()[1] * 0.9,
        f"{mean_resp:.1f}d", color="#e74c3c", fontsize=9)

plt.tight_layout()
plt.savefig("response_histogram.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"Figure saved → response_histogram.png")
print(f"Mean response time : {mean_resp:.1f} days")
print(f"Median response time: {resp_data.median():.1f} days")
print(f"Fastest response   : {resp_data.min():.0f} days")
print(f"Slowest response   : {resp_data.max():.0f} days")


### 🗺️ Month × Status Heatmap

A seaborn heatmap makes it easy to spot which month-status combinations
had the highest counts.  
Dark cells = many applications in that category.


In [ ]:
# ── Build month-ordered cross-tab ─────────────────────────────────────────────
month_order = ["January", "February", "March", "April"]   # Force chronological order

# Create cross-tabulation: rows = months, columns = statuses
heatmap_data = pd.crosstab(
    df["month_applied"],                            # Row variable
    df["status"],                                   # Column variable
).reindex(month_order)                              # Ensure chronological month order

# Reorder columns for logical reading (best → worst outcome)
col_order = [c for c in ["Offer","Interview","Applied","Rejected","Ghosted"]
             if c in heatmap_data.columns]
heatmap_data = heatmap_data[col_order]

# ── Plot heatmap ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

sns.heatmap(
    heatmap_data,
    annot=True,                                     # Show counts in each cell
    fmt="d",                                        # Integer format
    cmap="YlOrRd",                                  # Yellow → Orange → Red gradient
    linewidths=0.5,                                 # Grid lines between cells
    linecolor="white",                              # White grid lines
    cbar_kws={"label": "Count"},                    # Colour bar label
    ax=ax,
)
ax.set_title("Application Outcomes by Month (Heatmap)", fontsize=13)
ax.set_xlabel("Application Status")
ax.set_ylabel("Month Applied")
ax.tick_params(axis="x", rotation=15)               # Slight tilt for readability

plt.tight_layout()
plt.savefig("month_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()
print("Figure saved → month_heatmap.png")


---
## 🤖 Part 7 — Automated Weekly Report Generator

This is the automation centrepiece of the notebook.

The generator builds a complete markdown report from the live data
and saves it to `weekly_report.md`.  
In production, this script would run every Monday morning via a **cron job**:

```bash
# /etc/cron.d/job_report
0 8 * * 1   jupyter nbconvert --to notebook --execute job_tracker_automation.ipynb
```

### Report structure

| Section | Contents |
|---|---|
| Header | Report date, total apps, reporting period |
| KPI Summary | Offer rate, interview rate, response rate, ghosted rate |
| 🎉 Offers | List of companies that extended an offer |
| 💬 Active Interviews | Companies where interviews are in progress |
| 🔴 Overdue Follow-ups | Applications past their follow-up date |
| 📊 Platform Intelligence | Best-performing platform (by response rate) |
| 📅 Recent Activity | Applications submitted in the past 7 days |
| Footer | Auto-generated timestamp |


In [ ]:
# ── Gather data for the report ────────────────────────────────────────────────

# Offers list
offers_list = df[df["status"] == "Offer"][["company", "role", "date_applied"]].copy()
offers_list["date_applied"] = offers_list["date_applied"].dt.strftime("%b %d")

# Active interviews
interviews_active = df[df["status"] == "Interview"][
    ["company", "role", "interview_rounds", "date_applied"]
].copy()
interviews_active["date_applied"] = interviews_active["date_applied"].dt.strftime("%b %d")

# Overdue follow-ups (from Part 3 data)
overdue = df_active[df_active["alert"] == "OVERDUE"][
    ["company", "role", "follow_up_date", "days_to_followup"]
].copy()
overdue["days_overdue"] = overdue["days_to_followup"].abs()

# Best platform by response rate
best_platform = platform_stats.sort_values("response_rate", ascending=False).iloc[0]

# Recent applications (last 7 days)
recent_apps = df[df["days_since_applied"] <= 7].sort_values("date_applied", ascending=False)

print("Report data assembled.")
print(f"  Offers            : {len(offers_list)}")
print(f"  Active interviews : {len(interviews_active)}")
print(f"  Overdue follow-ups: {len(overdue)}")
print(f"  Recent apps (7d)  : {len(recent_apps)}")


### 🖊️ Build the Report String

We construct the report as a long multi-line Python string using f-string interpolation.
Each section is separated by a markdown horizontal rule (`---`).


In [ ]:
# ── Helper to format a DataFrame as a markdown table ─────────────────────────
def df_to_md_table(df_in, cols, headers=None):
    """Convert selected DataFrame columns to a markdown table string."""
    if headers is None:
        headers = cols                              # Use column names as headers
    col_widths = [max(len(h), df_in[c].astype(str).str.len().max())
                  for c, h in zip(cols, headers)]  # Column widths

    header_row = " | ".join(h.ljust(w) for h, w in zip(headers, col_widths))
    sep_row    = " | ".join("-" * w for w in col_widths)
    rows = []
    for _, row in df_in[cols].iterrows():
        rows.append(" | ".join(str(row[c]).ljust(w)
                               for c, w in zip(cols, col_widths)))

    return "\n".join(["| " + header_row + " |",
                       "| " + sep_row    + " |"] +
                      ["| " + r          + " |" for r in rows])

print("df_to_md_table() helper defined.")


In [ ]:
# ── Build the full markdown report string ────────────────────────────────────
report_date  = TODAY.strftime("%A, %B %d %Y")          # e.g. Wednesday, April 10 2024
week_start   = (TODAY - pd.Timedelta(days=7)).strftime("%b %d")
week_end     = TODAY.strftime("%b %d")

report = f"""# 📋 Weekly Job Application Report
**Generated:** {report_date}
**Reporting Period:** {week_start} – {week_end}
**Total Applications on File:** {len(df)}

---

## 📊 KPI Summary

| Metric | Value |
|---|---|
| Total Applications | {len(df)} |
| Offer Rate | {offer_rate:.1f}% |
| Interview Rate | {interview_rate:.1f}% |
| Response Rate | {response_rate:.1f}% |
| Ghosted Rate | {ghosted_rate:.1f}% |
| Avg Days to Response | {resp_data.mean():.1f} days |
| Overdue Follow-ups | {len(overdue)} |
| Active Interviews | {len(interviews_active)} |

---

## 🎉 Offers Received ({len(offers_list)})

"""

# ── Offers section ────────────────────────────────────────────────────────────
if len(offers_list) > 0:
    report += df_to_md_table(
        offers_list,
        cols    = ["company", "role", "date_applied"],
        headers = ["Company",  "Role",  "Applied Date"]
    )
else:
    report += "_No offers received yet. Keep applying!_"

report += f"""

---

## 💬 Active Interviews ({len(interviews_active)})

"""

if len(interviews_active) > 0:
    report += df_to_md_table(
        interviews_active,
        cols    = ["company", "role", "interview_rounds"],
        headers = ["Company",  "Role",  "Rounds"]
    )
else:
    report += "_No active interviews._"

report += f"""

---

## 🔴 Overdue Follow-ups ({len(overdue)})

These applications are past their suggested follow-up date.
Action required: send a polite follow-up email today.

"""

if len(overdue) > 0:
    report += df_to_md_table(
        overdue.head(10),
        cols    = ["company", "role", "days_overdue"],
        headers = ["Company",  "Role",  "Days Overdue"]
    )
else:
    report += "_No overdue follow-ups. Great job staying on top of things!_"

report += f"""

---

## 📊 Best Platform This Period

**{best_platform['platform']}** achieved the highest response rate at
**{best_platform['response_rate']:.1f}%** with an average salary of
**{best_platform['avg_salary_mid']}만원/month**.

Consider prioritising this platform for new applications next week.

---

## 📅 Applications in the Last 7 Days ({len(recent_apps)})

"""

if len(recent_apps) > 0:
    report += df_to_md_table(
        recent_apps[["company", "role", "status", "platform"]].head(15),
        cols    = ["company", "role", "status", "platform"],
        headers = ["Company",  "Role",  "Status", "Platform"]
    )
else:
    report += "_No applications in the past 7 days._"

report += f"""

---

## 💡 Recommendations for Next Week

1. **Follow up** on {len(overdue)} overdue applications — a brief email can reignite interest.
2. **Prioritise {best_platform['platform']}** — highest response rate in your data.
3. **Target new applications** at companies where referrals are possible (referral apps have higher offer rates).
4. **Prepare** for {len(interviews_active)} active interview processes.

---

_Report auto-generated by `job_tracker_automation.ipynb` on {report_date}_
_Ubuntu Automation for Data Science — Group Project_
"""

print("Report string built successfully.")
print(f"Report length: {len(report)} characters / ~{len(report.splitlines())} lines")


### 💾 Save & Print the Report

We save the report to `weekly_report.md` and also print it to the notebook output.
Printing it here lets you verify the content without opening the file.


In [ ]:
# ── Save the report to disk ───────────────────────────────────────────────────
REPORT_PATH = "weekly_report.md"                    # Output file path

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report)                                 # Write the full report string

print(f"Report saved to: {REPORT_PATH}")
print(f"File size: {os.path.getsize(REPORT_PATH):,} bytes")
print()


In [ ]:
# ── Print the full report to notebook output ─────────────────────────────────
print("=" * 70)
print("  WEEKLY REPORT PREVIEW (also saved as weekly_report.md)")
print("=" * 70)
print()
print(report)                                       # Print every line of the report
print()
print("=" * 70)
print("  END OF REPORT")
print("=" * 70)


---
## 🧠 Reflection

Working on this Job/Internship Application Tracker scenario gave me a much deeper
appreciation for how automation can turn raw spreadsheet data into actionable insights.

The most technically challenging part was **Part 3 — the Follow-up Alert System**.
Computing date differences in pandas is straightforward once you know to call `.dt.days`,
but handling the edge case where `follow_up_date` might itself be `NaT`
(if the original CSV had a blank) required careful use of `pd.Timestamp` arithmetic
and defaulting gracefully rather than crashing.

**Part 7 — the Automated Report Generator** was the most rewarding.
Seeing a multi-page markdown report assembled from live data felt genuinely useful —
I could imagine running this as a cron job every Monday morning and receiving
a Slack notification with the report content.
The `df_to_md_table()` helper I wrote is something I'll likely reuse in other projects.

The most surprising finding in the data was how large the **referral advantage** is:
applications with a referral consistently showed higher interview and offer rates
even though we simulated only a 20% referral rate in the dataset.
This reinforces the real-world advice to invest in networking, not just cold applications.

**Skills practised:**
- Pandas date arithmetic (`pd.Timestamp`, `.dt.days`, `isocalendar()`)
- Multi-figure matplotlib layouts (`plt.subplots` with `sharey`, nested helpers)
- Seaborn heatmaps and boxplots
- Python f-strings for report generation
- Modular function design (`funnel()`, `alert_level()`, `df_to_md_table()`)
- File I/O for saving markdown reports

If I were to extend this project, I would add **email delivery** using Python's
`smtplib` module so the weekly report lands in my inbox automatically,
and connect it to a real ATS API (like Notion or Airtable) via their REST APIs.

---
*End of notebook — Ubuntu Automation for Data Science, Group Project, Scenario 5*
